# Laya run 4: does the boundary pull the verdict down? (Kaggle, GPU T4 ×2)

Fourth run, on the **`laya_v2`** dataset (new version: equal passage split + the `v1` control folder). Same labels,
lectures and split as runs 1–3.

**What run 3 showed** (`notebooks/kaggle/finished/wahy-finetuning-laya-v2.ipynb`, `docs/reports/laya-run3-report.md`):
* the short-verse fix works where it was aimed: verses inside a passage, real ones rejected 3/64 → **0/64**;
* but every v2 model is **worse on the verdict** than run 2's `curve_100` (same 2 epochs, old envelope): AUC on your cards
  0.945 → 0.857–0.890, *not-a-quote* caught 85 % → 48–70 %;
* and the yes/no boundary collapsed to "never cut off" (balanced 0.5), except `both_synth` (0.60 / 0.59, still under the
  no-model rule's 0.69 / 0.75).

**The hypothesis:** the boundary questions compete with the verdict inside the one shared model. Run 2 already hinted at
it (more boundary items → lower verdict AUC: 0.945 → 0.93 → 0.918); run 3 doubled the boundary items (two questions per
real quote, "yes" × 3) and the model with the most of them (`both_synth`) has the lowest verdict.

| run | data | trains on | answers |
|---|---|---|---|
| `both_verdict` | v2, passages + neighbours | **verdict only** | without the boundary, does the verdict come back — and keep the short-verse gains? |
| `v1_verdict` | old envelope (laya_v1) | **verdict only** | the control: is it the boundary competing, or the v2 input itself? |
| `boundary_only` | v2 + counterfactual rows | **start / end cut off only** | can Laya learn the boundary when it is its only job? |
| `both` | v2, passages + neighbours | all three (= run 3's `both`, equal split) | how much do numbers move between two identical runs (noise)? |

Also new: the boundary is scored a second time with **thresholds tuned on val** (a model can rank cut-off quotes well and
still never reach 0.5: run 3's AUCs were 0.6–0.75 with balanced 0.5).

**If `both_verdict` wins**, the app is ready: a checkpoint trained on the verdict only carries `wahy_questions: ["verdict"]`,
the app asks it only that (3× fewer readings per finding) and takes the boundary from the word-overlap rule — the best
boundary so far. A boundary-only model is for learning here; the app does not run two Laya models.

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets:** `HF_TOKEN`
* **Input:** the `laya_v2` dataset, **new version** made from the rebuilt `data/laya_datagen/laya_v2.zip` (section 3 checks it)
* **Run:** *Save Version → Save & Run All (Commit)*. Expect **~2 h** (4 trainings, 2 of them half-size).
* **Download afterwards:** the `results/` folder → `data/laya_datagen/results/run4/` → `notebooks/results/laya_v2_compare.ipynb`.

## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q -U "laya>=0.4.12,<0.5" "transformers>=4.48.0" safetensors huggingface_hub pandas matplotlib tabulate
import laya, transformers
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Dataset, scripts, base checkpoint, Hub login
Stops early if the dataset is the run-3 version (no `v1` folder, no `--questions` in `train_ddp.py`).

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download, login
from laya.agent import _fix_tokenizer_config

DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v2")
manifest = json.loads((DATA / "manifest.json").read_text())
assert manifest["envelope"] == 2 and "v1" in manifest["variants"], (
    "This is the run-3 version of laya_v2: upload the rebuilt laya_v2.zip as a NEW VERSION of the dataset, then re-run.")
assert "--questions" in (DATA / "train_ddp.py").read_text() and "def tune_thresholds" in (DATA / "eval_laya.py").read_text(), (
    "Old scripts in the dataset: upload the rebuilt laya_v2.zip as a NEW VERSION of the dataset, then re-run.")
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest["created"], "| base model", manifest["base_model"],
      "| max_len", manifest["max_len"], "| variants", list(manifest["variants"]))
MODEL_ID = manifest["base_model"]          # convaiinnovations/laya-multilingual
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Hub: logged in")
except Exception as e:
    HF_TOKEN = None
    print("Hub: no HF_TOKEN secret, the 'before' line (private model) and the push will be skipped:", e)

## 4. Settings
Run 3's recipe for everything not in `RUNS`: 2 epochs, effective batch 64 (4 × 8 accumulation), RLCD, the same learning
rates, max_len 2048, a "yes, cut off" answer × 3 in the loss.

In [ ]:
RUNS = {   # name -> (data folder, questions trained, what it answers)
    "both_verdict":  ("both", ["verdict"], "v2 + D + A, verdict only"),
    "v1_verdict":    ("v1", ["verdict"], "old envelope, verdict only (control)"),
    "boundary_only": ("both_synth", ["start_missing", "end_missing"], "v2 + D + A + counterfactual, boundary only"),
    "both":          ("both", None, "= run 3's both, all questions (noise check)"),
}
NOUL_TRUE_WEIGHT = 3.0
TRAIN_ARGS = (f"--epochs 2 --micro-batch 4 --grad-accum 8 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd "
              f"--max-len {manifest['max_len']} --head-max-len {manifest['head_max_len']} --noul-true-weight {NOUL_TRUE_WEIGHT}")
APP_FALSE_SHARE = 0.10
BEFORE_MODEL = "ZenoAbderrahim/laya-wahy-soft_asis"     # the app's current model: the "before" line
PUSH_TO_HUB = True
HUB_PREFIX = "ZenoAbderrahim/laya-wahy-r4"              # repo = f"{HUB_PREFIX}-{run}" (run 3's repos stay untouched)

WORK = Path("/kaggle/working")
for d in ("results", "logs"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

def run(cmd, log):
    # Run a shell command, stream its output here and into a log file; returns the exit code.
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

import pandas as pd
rows = []
for name, (data, qs, about) in RUNS.items():
    tr = manifest["variants"][data]["train"]
    items = (tr["rows"] if not qs or "verdict" in qs else 0) + sum(tr.get(f"{q} items", 0) for q in ("start_missing", "end_missing") if not qs or q in qs)
    rows.append({"run": name, "data": data, "questions": ", ".join(qs) if qs else "all", "about": about,
                 "train rows": tr["rows"], "train items": items, "not a quote (rows)": round(tr.get("verdict FALSE", 0) / tr["rows"], 3)})
pd.DataFrame(rows).set_index("run")

## 5. The "before" line: your current app model on the v1 test file
Same as run 3, plus its boundary thresholds tuned on the v1 val file.

In [ ]:
flags = f"--verse-flags {DATA}/test_verse_flags.json"
if HF_TOKEN:
    rc = run(f"python {DATA}/eval_laya.py predict --model {BEFORE_MODEL} --name 'before: soft_asis (app)' "
             f"--test {DATA}/v1_test.jsonl --out {WORK}/results/before_soft_asis.json --app-false-share {APP_FALSE_SHARE} "
             f"--batch-size 32 --baselines {DATA}/v1_baselines.json --tune-val {DATA}/v1/val.jsonl {flags}",
             WORK / "logs" / "eval_before.log")
    print("before:", "ok" if rc == 0 else f"failed (exit {rc}), see logs/eval_before.log")

## 6. Train + evaluate
`--questions` keeps only those items in train and val; the checkpoint's config records them (`wahy_questions`) so the
evaluation and the app ask it nothing else. Models with a boundary get `--tune-val` (thresholds from their val file).

In [ ]:
import time
status = {}
for name, (data, qs, _) in RUNS.items():
    v = manifest["variants"][data]
    out = WORK / "models" / name
    t0 = time.time()
    try:
        extra = dict(v["cfg_extra"], **({"wahy_questions": qs} if qs else {}))
        cfg_extra = json.dumps(extra).replace('"', '\\"')
        qarg = f"--questions {','.join(qs)}" if qs else ""
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {DATA}/{data}/train.jsonl --val {DATA}/{data}/val.jsonl --name laya-wahy-r4-{name} "
                 f"--cfg-extra \"{cfg_extra}\" {qarg} {TRAIN_ARGS}", WORK / "logs" / f"train_{name}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[name] = f"training failed (exit {rc}), see logs/train_{name}.log"
            continue
        tr = v["train"]
        tune = f"--tune-val {DATA}/{data}/val.jsonl" if (not qs or "start_missing" in qs) else ""
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {name} --test {DATA}/{data}/test.jsonl "
                 f"--out {WORK}/results/{name}.json --train-false-share {round(tr.get('verdict FALSE', 0) / tr['rows'], 4)} "
                 f"--app-false-share {APP_FALSE_SHARE} --batch-size 16 {tune} {flags}", WORK / "logs" / f"eval_{name}.log")
        status[name] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
        train_log = out / "train_log.json"
        if train_log.exists():
            (WORK / "results" / f"train_log_{name}.json").write_text(train_log.read_text())
    except Exception as e:
        status[name] = f"error: {e}"
    print(f"\n##### {name}: {status[name]}\n", flush=True)
json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Load the results (+ the no-model boundary rule)

In [ ]:
from IPython.display import Markdown, display
v1_rows = eval_laya.read_jsonl(DATA / "v1_test.jsonl")
vflags = json.loads((DATA / "test_verse_flags.json").read_text())
res = {n: json.load(open(WORK / "results" / f"{n}.json")) for n in RUNS if (WORK / "results" / f"{n}.json").exists()}
before = [json.load(open(WORK / "results" / "before_soft_asis.json"))] if (WORK / "results" / "before_soft_asis.json").exists() else []
rule = eval_laya.evaluate(*eval_laya.per_verse(v1_rows, eval_laya.rule_predictions(v1_rows), vflags), "word-overlap rule (no model)")
json.dump(rule, open(WORK / "results" / "rule_boundary.json", "w"), ensure_ascii=False)
teachers = [json.load(open(f)) for f in sorted(glob.glob(str(WORK / "results" / "baseline_*.json")))]
models = before + [res[n] for n in RUNS if n in res]
# run 2's curve_100 (old envelope, all questions, 2 epochs) on the same cards, for the verdict reference line
CURVE_100 = {"you": {"AUC": 0.945, "accuracy": 0.894, "not-a-quote recall": 0.852, "real rejected": 0.099},
             "you + Sonnet": {"AUC": 0.954, "accuracy": 0.899, "not-a-quote recall": 0.947, "real rejected": 0.142}}
print("models:", [m["name"] for m in models], "| teacher baselines:", [t["name"] for t in teachers])

## 8. Verdict
The question of this run: `both_verdict` and `v1_verdict` against `both` (and run 2's `curve_100`: AUC on your cards
0.945, on you + Sonnet 0.954). Then the slices the passages / neighbours are for.

In [ ]:
verdict_models = [m for m in models if any((g or {}).get("verdict") for g in m["groups"].values())]
report = []
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    t = f"### verdict · {g}\n" + eval_laya.compare(verdict_models + teachers, g)
    report.append(t); display(Markdown(t))
for g in ["you", "you + Sonnet"]:
    for part in ("by context", "by span length"):
        t = f"### verdict {part} · {g}\n" + eval_laya.compare_slices(verdict_models, g, part)
        report.append(t); display(Markdown(t))
(WORK / "results" / "comparison.md").write_text("\n\n".join(report))

## 9. Boundary — at 0.5, then with thresholds tuned on val
**Balanced accuracy** is the column to read. The second table cuts each model's P(cut off) where its val file says,
which is what the app would do with that model.

In [ ]:
boundary_models = [m for m in models if any((g or {}).get("boundary") for g in m["groups"].values())]
tuned = [dict(m["tuned"], name=m["tuned"]["name"]) for m in boundary_models if m.get("tuned")]
btext = []
for title, ms in (("at 0.5", boundary_models), ("thresholds tuned on val", tuned)):
    for g in ["you", "written text", "you + Sonnet"]:
        t = f"### boundary {title} · {g}\n" + eval_laya.compare_boundary([rule] + (teachers if title == "at 0.5" else []) + ms, g)
        btext.append(t); display(Markdown(t))
(WORK / "results" / "boundary.md").write_text("\n\n".join(btext))
print("thresholds:", {m["name"]: m.get("thresholds") for m in boundary_models})

logs = {n: json.load(open(WORK / "results" / f"train_log_{n}.json")) for n in RUNS if (WORK / "results" / f"train_log_{n}.json").exists()}
display(pd.DataFrame([{"run": n, **{f"ep{e['epoch']} {k}": e["val"][k].get("balanced_accuracy", round(e["val"][k]["accuracy"], 4))
                                    for e in logs[n]["epochs"] for k in e["val"] if k != "all"}}
                      for n in logs]).set_index("run"))

## 10. One picture

In [ ]:
import matplotlib.pyplot as plt

def val(m, group, part, key, src="groups"):
    x = ((m.get(src) or {}).get("groups", m.get(src)) if src != "groups" else m["groups"])
    x = ((x or {}).get(group) or {}).get(part) or {}
    x = x.get(key)
    return x[0] if isinstance(x, list) else x

def slice_rej(m, group, name):
    for sl in ((m["groups"].get(group) or {}).get("verdict") or {}).get("by context", []):
        if sl["slice"] == name:
            return sl["real quotes wrongly rejected"][0]

panels = [("you · verdict AUC\n(dashed: run 2 curve_100)", lambda m: val(m, "you", "verdict", "AUC P(not a quote)"), CURVE_100["you"]["AUC"]),
          ("you · not-a-quote recall", lambda m: val(m, "you", "verdict", "not-a-quote recall"), CURVE_100["you"]["not-a-quote recall"]),
          ("you+Sonnet · verdict AUC", lambda m: val(m, "you + Sonnet", "verdict", "AUC P(not a quote)"), CURVE_100["you + Sonnet"]["AUC"]),
          ("you+Sonnet · in a passage:\nreal wrongly rejected", lambda m: slice_rej(m, "you + Sonnet", "inside a passage of consecutive verses"), None),
          ("you · boundary balanced acc\n(tuned on val)", lambda m: val(m, "you", "boundary", "balanced accuracy", "tuned"), None)]
names = [m["name"] for m in models]
fig, ax = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.8))
for a, (title, f, ref) in zip(ax, panels):
    ys = [f(m) for m in models]
    a.bar(range(len(models)), [y if y is not None else 0 for y in ys], color=["#888"] + ["C0"] * (len(models) - 1))
    if ref is not None:
        a.axhline(ref, ls="--", c="grey", lw=1)
    a.set_xticks(range(len(models)), names, rotation=45, ha="right", fontsize=8); a.set_title(title, fontsize=9); a.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(WORK / "results" / "run4.png", dpi=120); plt.show()

## 11. (Optional) Push the models to the Hugging Face Hub — private
`both_verdict` / `v1_verdict` work in the app as they are (verdict from the model, boundary from the rule).
`boundary_only` answers no verdict: keep it for study, do not load it in the app.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for n in RUNS:
        out = WORK / "models" / n
        if not (out / "model.safetensors").exists():
            continue
        repo = f"{HUB_PREFIX}-{n}"
        api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo, repo_type="model",
                          commit_message=f"laya-multilingual fine-tuned on Wahy laya_v2 (run 4: {n}, {RUNS[n][2]})")
        print("pushed", repo)